In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset , DataLoader
import torch.nn as nn

In [2]:
torch.manual_seed(42)

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
import kagglehub
path = kagglehub.dataset_download("zalando-research/fashionmnist")

train_path = "/kaggle/input/fashionmnist/fashion-mnist_train.csv"
df = pd.read_csv(train_path)

Using Colab cache for faster access to the 'fashionmnist' dataset.


In [5]:
df.sample()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
17350,1,0,0,0,0,0,0,0,0,0,...,188,124,0,0,0,0,0,0,0,0


In [6]:
X = df.drop('label',axis=1)
y = df['label']

In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
# transfromation
from torchvision.transforms import transforms

custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [10]:
from PIL import Image

In [27]:
class CustomData(Dataset):

  def __init__(self,features,labels,transform):
    self.features = features
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.features)

  def __getitem__(self,idx):
    # resize 28 * 28
    image = self.features.iloc[idx].values.reshape(28,28)

    # change dytpe in to np.uint8
    image = image.astype(np.uint8)

    # change black and white to colured
    image = np.stack((image,)*3,axis=-1)

    # convert array to PIL
    image = Image.fromarray(image)

    # apply transform
    image = self.transform(image)

    # return
    return image,torch.tensor(self.labels.iloc[idx],dtype=torch.long)

In [28]:
train_dataset = CustomData(X_train,y_train,custom_transform)
test_dataset = CustomData(X_test,y_test,custom_transform)

In [29]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [17]:
# fecth pretrained model
import torchvision.models as models

vgg16 = models.vgg16(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:08<00:00, 69.2MB/s]


In [19]:
vgg16.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [30]:
for param in vgg16.features.parameters():
  param.requires_grad = False

In [31]:
vgg16.classifier = nn.Sequential(
    nn.Linear(in_features=25088, out_features=1024, bias=True),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.5, inplace=False),
    nn.Linear(in_features=1024, out_features=512, bias=True),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.5, inplace=False),
    nn.Linear(in_features=512, out_features=10, bias=True)
)

In [21]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [32]:
vgg16 = vgg16.to(device)

In [37]:
learning_rate = 0.0001
epochs = 5

In [38]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(vgg16.classifier.parameters(),lr=learning_rate)

In [39]:
for epoch in range(epochs):

  total_loss = 0

  for batch_features,batch_labels in train_loader:
    # move to gpu
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)

    # forward pass
    predications = vgg16(batch_features)

    # calculate loss
    loss = criterion(predications,batch_labels)

    # make grad zero
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # update parameter
    optimizer.step()

    total_loss += loss.item()


  print(f'epoch: {epoch+1} avg_loss: {total_loss/len(train_loader)}')

epoch: 1 avg_loss: 0.2509778426873187
epoch: 2 avg_loss: 0.1841422681224843
epoch: 3 avg_loss: 0.14483307850391913
epoch: 4 avg_loss: 0.11255367199967926
epoch: 5 avg_loss: 0.09098147135569404


In [41]:
 # set model to eval mode
vgg16.eval()

total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)

    outputs = vgg16(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.9253333333333333
